# NB4 — End-to-End Prediction Pipeline — FINAL v4
### Novel ABSA on Voice — Final Year Project

**Changes in v4:**
- ✅ All v3 functionality preserved (Whisper medium, M4A support, BIO aspect extraction, DeBERTa ABSA)
- 🎵 **NEW:** Predicted tone (emotion from voice) now shown in output alongside ABSA results

```
WAV / M4A
  ↓
[0] M4A → WAV conversion (if needed, via pydub)
  ↓
[1] Whisper medium       → accurate transcript
  ↓                      ↓ (parallel)
[2] BIO Model            wav2vec2 → emotion → tone sentiment
  ↓
[3] DeBERTa ABSA         → sentiment per aspect
  ↓
[4] Full report (ABSA + Tone)
```

**Prerequisites:** NB0 + NB2 + NB1 trained models saved to Drive.

In [1]:
# CELL 1 — Install
!pip install -q transformers==4.40.0 accelerate==0.29.3 openai-whisper pydub
!apt-get install -y -q ffmpeg  # needed by pydub for m4a conversion
print('✅ Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 wh

In [3]:
# CELL 2 — Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, gc, json, warnings
import numpy as np
import torch
import torch.nn.functional as F
warnings.filterwarnings('ignore')

BASE           = '/content/drive/MyDrive/Final year project/THE FINAL PROBLEM'
ASP_MODEL_DIR  = os.path.join(BASE, 'aspect_extractor/best_model')
TEXT_MODEL_DIR = os.path.join(BASE, 'deberta_absa/best_model')
EMO_MODEL_DIR  = os.path.join(BASE, 'emotion_model/best_model')   # NB1 wav2vec2
LOCAL_ASP      = '/content/asp_local'
LOCAL_TEXT     = '/content/text_local'
LOCAL_EMO      = '/content/emo_local'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ID2LABEL_SENTIMENT   = {0:'negative', 1:'neutral', 2:'positive'}
ID2LABEL_BIO         = {0:'O', 1:'B-ASP', 2:'I-ASP'}
ID2LABEL_EMOTION     = {0:'neutral', 1:'happy', 2:'sad', 3:'angry', 4:'fearful', 5:'disgust'}
EMOTION_TO_SENTIMENT = {'happy':2, 'neutral':1, 'sad':1, 'angry':0, 'fearful':0, 'disgust':0}
EMOTION_EMOJI        = {'happy':'😄','neutral':'😐','sad':'😢','angry':'😠','fearful':'😨','disgust':'🤢'}

for name, path in [('NB0 aspect extractor', ASP_MODEL_DIR),
                   ('NB2 DeBERTa ABSA',     TEXT_MODEL_DIR),
                   ('NB1 wav2vec2 emotion',  EMO_MODEL_DIR)]:
    print(f'{name}: {"✅" if os.path.exists(path) else "❌ NOT FOUND"}  {path}')
print(f'\n✅ Device: {DEVICE}')

Mounted at /content/drive
NB0 aspect extractor: ✅  /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/aspect_extractor/best_model
NB2 DeBERTa ABSA: ✅  /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/deberta_absa/best_model
NB1 wav2vec2 emotion: ✅  /content/drive/MyDrive/Final year project/THE FINAL PROBLEM/emotion_model/best_model

✅ Device: cuda


In [4]:
# CELL 3 — Copy models to local disk
if not os.path.exists(LOCAL_ASP):
    print('Copying aspect extractor...')
    shutil.copytree(ASP_MODEL_DIR, LOCAL_ASP); print('✅')
else:
    print('✅ Aspect extractor already local')

if not os.path.exists(LOCAL_TEXT):
    print('Copying DeBERTa...')
    shutil.copytree(TEXT_MODEL_DIR, LOCAL_TEXT); print('✅')
else:
    print('✅ DeBERTa already local')

if not os.path.exists(LOCAL_EMO):
    print('Copying wav2vec2 emotion model...')
    shutil.copytree(EMO_MODEL_DIR, LOCAL_EMO); print('✅')
else:
    print('✅ Emotion model already local')

Copying aspect extractor...
✅
Copying DeBERTa...
✅
Copying wav2vec2 emotion model...
✅


In [5]:
# CELL 4 — Load Whisper MEDIUM
import whisper
print('Loading Whisper medium (first time downloads ~1.4GB)...')
whisper_model = whisper.load_model('medium')
print('✅ Whisper medium loaded')

Loading Whisper medium (first time downloads ~1.4GB)...


100%|██████████████████████████████████████| 1.42G/1.42G [00:10<00:00, 145MiB/s]


✅ Whisper medium loaded


In [6]:
# CELL 5 — Load aspect extractor
from transformers import AutoTokenizer, AutoModelForTokenClassification
asp_tok = AutoTokenizer.from_pretrained(LOCAL_ASP)
asp_mdl = AutoModelForTokenClassification.from_pretrained(LOCAL_ASP).to(DEVICE).eval()
print(f'✅ Aspect extractor loaded | Labels: {asp_mdl.config.id2label}')

✅ Aspect extractor loaded | Labels: {0: 'O', 1: 'B-ASP', 2: 'I-ASP'}


In [7]:
# CELL 6 — Load DeBERTa ABSA
from transformers import AutoModelForSequenceClassification
txt_tok = AutoTokenizer.from_pretrained(LOCAL_TEXT)
txt_mdl = AutoModelForSequenceClassification.from_pretrained(LOCAL_TEXT).to(DEVICE).eval()
print('✅ DeBERTa ABSA loaded')

✅ DeBERTa ABSA loaded


In [8]:
# CELL 6b — Load wav2vec2 emotion model (NB1)  [NEW in v4]
import json as _json
import torchaudio
from transformers import Wav2Vec2FeatureExtractor, AutoModelForAudioClassification

# Fix added_tokens.json if empty (known issue)
for _mdir in [LOCAL_ASP, LOCAL_TEXT, LOCAL_EMO]:
    _at = os.path.join(_mdir, 'added_tokens.json')
    if os.path.exists(_at):
        with open(_at) as _f: _c = _f.read().strip()
        if not _c:
            with open(_at, 'w') as _f: _json.dump({}, _f)
            print(f'🔧 Fixed empty added_tokens.json in {os.path.basename(_mdir)}')

emo_feat = Wav2Vec2FeatureExtractor.from_pretrained(LOCAL_EMO)
emo_mdl  = AutoModelForAudioClassification.from_pretrained(LOCAL_EMO).to(DEVICE).eval()
print(f'✅ wav2vec2 emotion model loaded | Labels: {emo_mdl.config.id2label}')

✅ wav2vec2 emotion model loaded | Labels: {0: 'neutral', 1: 'happy', 2: 'sad', 3: 'angry', 4: 'fearful', 5: 'disgust'}


In [9]:
# CELL 7 — M4A → WAV converter
from pydub import AudioSegment
import tempfile

def convert_to_wav(input_path: str) -> str:
    ext = os.path.splitext(input_path)[1].lower()
    if ext == '.wav':
        print('  Already WAV — no conversion needed')
        return input_path
    print(f'  Converting {ext} → WAV...')
    audio = AudioSegment.from_file(input_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.wav')
    audio.export(tmp.name, format='wav')
    print(f'  ✅ Converted → {tmp.name} ({len(audio)/1000:.1f}s)')
    return tmp.name

print('✅ convert_to_wav() ready')

✅ convert_to_wav() ready


In [10]:
# CELL 8 — Aspect extraction function
def extract_aspects(text: str, max_len: int = 128) -> list:
    enc = asp_tok(text, max_length=max_len, truncation=True,
                   return_offsets_mapping=True, return_tensors='pt')
    offsets = enc['offset_mapping'][0].tolist()
    inp = {k: v.to(DEVICE) for k, v in enc.items() if k != 'offset_mapping'}
    with torch.no_grad():
        preds = asp_mdl(**inp).logits.argmax(-1)[0].cpu().tolist()

    aspects = []
    cur_start = cur_end = None
    for label_id, (cs, ce) in zip(preds, offsets):
        label = ID2LABEL_BIO[label_id]
        if cs == 0 and ce == 0:
            if cur_start is not None:
                span = text[cur_start:cur_end].strip()
                if span: aspects.append(span)
                cur_start = cur_end = None
            continue
        if label == 'B-ASP':
            if cur_start is not None:
                span = text[cur_start:cur_end].strip()
                if span: aspects.append(span)
            cur_start, cur_end = cs, ce
        elif label == 'I-ASP':
            if cur_start is not None: cur_end = ce
            else: cur_start, cur_end = cs, ce
        else:
            if cur_start is not None:
                span = text[cur_start:cur_end].strip()
                if span: aspects.append(span)
                cur_start = cur_end = None
    if cur_start is not None:
        span = text[cur_start:cur_end].strip()
        if span: aspects.append(span)

    seen, unique = set(), []
    for a in aspects:
        if a.lower() not in seen:
            seen.add(a.lower()); unique.append(a)
    return unique

print('✅ extract_aspects() ready')

✅ extract_aspects() ready


In [11]:
# CELL 9 — Sentiment classification function
def classify_aspects(transcript: str, aspects: list) -> list:
    results = []
    for aspect in aspects:
        enc = txt_tok(transcript, aspect, return_tensors='pt',
                      max_length=128, truncation=True, padding='max_length')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            probs = torch.softmax(txt_mdl(**enc).logits, dim=-1).squeeze(0).cpu().numpy()
            pred  = int(probs.argmax())
            conf  = float(probs.max())
        results.append({
            'aspect':     aspect,
            'sentiment':  ID2LABEL_SENTIMENT[pred],
            'confidence': round(conf, 3),
            'probs': {
                'negative': round(float(probs[0]), 3),
                'neutral':  round(float(probs[1]), 3),
                'positive': round(float(probs[2]), 3),
            }
        })
    results.sort(key=lambda x: x['confidence'], reverse=True)
    return results

print('✅ classify_aspects() ready')

✅ classify_aspects() ready


In [12]:
# CELL 9b — Tone prediction function  [NEW in v4]
def predict_tone(wav_path: str) -> dict:
    """Run NB1 wav2vec2 emotion model on WAV and return tone prediction."""
    waveform, sr = torchaudio.load(wav_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    if waveform.shape[0] > 1:              # stereo → mono
        waveform = waveform.mean(0, keepdim=True)
    waveform = waveform.squeeze(0).numpy()

    inputs = emo_feat(waveform, sampling_rate=16000,
                      return_tensors='pt', padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(emo_mdl(**inputs).logits, dim=-1).squeeze(0).cpu().numpy()

    pred_id       = int(probs.argmax())
    pred_emotion  = ID2LABEL_EMOTION[pred_id]
    tone_sent_id  = EMOTION_TO_SENTIMENT[pred_emotion]
    return {
        'emotion':        pred_emotion,
        'tone_sentiment': ID2LABEL_SENTIMENT[tone_sent_id],
        'confidence':     round(float(probs.max()), 3),
        'emotion_probs':  {ID2LABEL_EMOTION[i]: round(float(p), 3)
                           for i, p in enumerate(probs)}
    }

print('✅ predict_tone() ready')

✅ predict_tone() ready


In [13]:
# CELL 10 — Full pipeline function
def predict_all_aspects(audio_path: str):
    print('='*65)
    print(f'🎤 File: {os.path.basename(audio_path)}')
    print('='*65)
    wav_path = convert_to_wav(audio_path)

    print('\n[1/4] Transcribing with Whisper medium...')
    result = whisper_model.transcribe(
        wav_path,
        language='en',
        task='transcribe',
        fp16=torch.cuda.is_available(),
        verbose=False
    )
    transcript = result['text'].strip()
    print(f'  📝 Transcript: "{transcript}"')
    if not transcript:
        print('  ❌ Empty transcript')
        return

    print('\n[2/4] Detecting tone from voice...')
    try:
        tone = predict_tone(wav_path)
    except Exception as e:
        print(f'  ⚠️  Tone detection failed: {e}')
        tone = None

    print('\n[3/4] Extracting aspects...')
    aspects = extract_aspects(transcript)
    if not aspects:
        print('  ⚠️  No aspects found.')
        return {'transcript': transcript, 'tone': tone, 'aspects': []}
    print(f'  🎯 Found {len(aspects)} aspect(s): {aspects}')

    print('\n[4/4] Classifying sentiment per aspect...')
    results = classify_aspects(transcript, aspects)

    # ── ABSA Report ──────────────────────────────────────────────────
    print('\n' + '='*65)
    print('🧠 ASPECT-SENTIMENT REPORT')
    print('='*65)
    print(f'{"Aspect":<25} {"Sentiment":<12} {"Conf":<8} neg/neu/pos')
    print('-'*65)
    for r in results:
        p = r['probs']
        e = {'positive':'🟢','neutral':'🟡','negative':'🔴'}[r['sentiment']]
        print(f'{r["aspect"]:<25} {e} {r["sentiment"]:<10} '
              f'{r["confidence"]:.3f}    '
              f'{p["negative"]:.2f}/{p["neutral"]:.2f}/{p["positive"]:.2f}')
    print('='*65)

    # ── Tone Report  [NEW in v4] ──────────────────────────────────────
    print('\n' + '━'*65)
    print('🎵 PREDICTED TONE  (from voice)')
    print('━'*65)
    if tone:
        emo_e  = EMOTION_EMOJI.get(tone['emotion'], '❓')
        sent_e = {'positive':'🟢','neutral':'🟡','negative':'🔴'}[tone['tone_sentiment']]
        print(f'  Detected Emotion   : {emo_e}  {tone["emotion"].upper()}  (conf: {tone["confidence"]:.3f})')
        print(f'  Tone Sentiment     : {sent_e}  {tone["tone_sentiment"].upper()}')
        print(f'  Emotion breakdown  :')
        for emo, prob in tone['emotion_probs'].items():
            bar = '█' * int(prob * 20)
            print(f'    {emo:<10} {bar:<20} {prob:.3f}')
    else:
        print('  ⚠️  Tone not available')
    print('━'*65)

    if wav_path != audio_path and os.path.exists(wav_path):
        os.remove(wav_path)
    return {'transcript': transcript, 'tone': tone, 'aspects': results}

print('✅ predict_all_aspects() ready')

✅ predict_all_aspects() ready


In [14]:
# CELL 11 — Test with raw text
def predict_from_text(sentence: str):
    print('='*65)
    print(f'📝 Input: "{sentence}"')
    print('='*65)
    aspects = extract_aspects(sentence)
    print(f'🎯 Aspects ({len(aspects)}): {aspects}')
    if not aspects:
        print('No aspects found.'); return
    results = classify_aspects(sentence, aspects)
    print(f'\n{"Aspect":<25} {"Sentiment":<12} {"Conf":<8} neg/neu/pos')
    print('-'*65)
    for r in results:
        p = r['probs']
        e = {'positive':'🟢','neutral':'🟡','negative':'🔴'}[r['sentiment']]
        print(f'{r["aspect"]:<25} {e} {r["sentiment"]:<10} '
              f'{r["confidence"]:.3f}    '
              f'{p["negative"]:.2f}/{p["neutral"]:.2f}/{p["positive"]:.2f}')
    print('='*65)
    print('  ℹ️  Tone N/A — voice input required for tone prediction')
    return results

tests = [
    'The pizza was amazing but the service was really slow.',
    'Great food and friendly staff but prices are too high and parking is terrible.',
    'The pasta was overcooked and the waiter was rude but the dessert was fantastic.',
]
for t in tests:
    predict_from_text(t)
    print()

📝 Input: "The pizza was amazing but the service was really slow."
🎯 Aspects (2): ['pizza', 'service']

Aspect                    Sentiment    Conf     neg/neu/pos
-----------------------------------------------------------------
pizza                     🟢 positive   1.000    0.00/0.00/1.00
service                   🔴 negative   0.999    1.00/0.00/0.00
  ℹ️  Tone N/A — voice input required for tone prediction

📝 Input: "Great food and friendly staff but prices are too high and parking is terrible."
🎯 Aspects (4): ['food', 'staff', 'prices', 'parking']

Aspect                    Sentiment    Conf     neg/neu/pos
-----------------------------------------------------------------
food                      🟢 positive   1.000    0.00/0.00/1.00
staff                     🟢 positive   0.999    0.00/0.00/1.00
parking                   🔴 negative   0.999    1.00/0.00/0.00
prices                    🔴 negative   0.998    1.00/0.00/0.00
  ℹ️  Tone N/A — voice input required for tone prediction

📝 In

In [15]:
# CELL 12 — Upload WAV or M4A and predict
from google.colab import files
import traceback

print('Upload a .wav or .m4a audio file:')
uploaded = files.upload()

for fname, data in uploaded.items():
    tmp = f'/content/{fname}'
    with open(tmp, 'wb') as f: f.write(data)
    print(f'\n📁 Uploaded: {fname} ({len(data)/1024:.0f} KB)')
    try:
        result = predict_all_aspects(tmp)
        if result:
            print('\nFull JSON:')
            print(json.dumps(result, indent=2, default=float))
    except Exception:
        traceback.print_exc()

Upload a .wav or .m4a audio file:


Saving Recording (80).m4a to Recording (80).m4a

📁 Uploaded: Recording (80).m4a (258 KB)
🎤 File: Recording (80).m4a
  Converting .m4a → WAV...
  ✅ Converted → /tmp/tmpcp3be218.wav (10.8s)

[1/4] Transcribing with Whisper medium...


100%|██████████| 1075/1075 [00:01<00:00, 598.06frames/s]


  📝 Transcript: "The ambience was good, the stops were nice but the price was too much high."

[2/4] Detecting tone from voice...

[3/4] Extracting aspects...
  🎯 Found 3 aspect(s): ['ambience', 'stops', 'price']

[4/4] Classifying sentiment per aspect...

🧠 ASPECT-SENTIMENT REPORT
Aspect                    Sentiment    Conf     neg/neu/pos
-----------------------------------------------------------------
ambience                  🟢 positive   1.000    0.00/0.00/1.00
stops                     🟢 positive   1.000    0.00/0.00/1.00
price                     🔴 negative   0.999    1.00/0.00/0.00

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎵 PREDICTED TONE  (from voice)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Detected Emotion   : 🤢  DISGUST  (conf: 0.800)
  Tone Sentiment     : 🔴  NEGATIVE
  Emotion breakdown  :
    neutral                         0.017
    happy      █                    0.065
    sad                             0.036
    a

In [16]:
# CELL 13 — Test M4A → WAV conversion
import tempfile, struct
from pydub import AudioSegment
from pydub.generators import Sine

print('Testing M4A → WAV conversion...')
tone = Sine(440).to_audio_segment(duration=2000)
test_m4a = '/content/test_tone.m4a'
tone.export(test_m4a, format='mp4')
print(f'Created test M4A: {os.path.getsize(test_m4a)} bytes')
wav = convert_to_wav(test_m4a)
print(f'Converted WAV: {wav}')
print(f'WAV size: {os.path.getsize(wav)} bytes')
print('✅ M4A → WAV conversion works!')
os.remove(test_m4a)
os.remove(wav)

Testing M4A → WAV conversion...
Created test M4A: 18835 bytes
  Converting .m4a → WAV...
  ✅ Converted → /tmp/tmpg3m260mj.wav (2.0s)
Converted WAV: /tmp/tmpg3m260mj.wav
WAV size: 64688 bytes
✅ M4A → WAV conversion works!
